In [1]:
print("hello")

hello


In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import os
import random

# ==========================================
# 0. GPU 및 시드 고정 설정
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 중인 디바이스: {device}")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ==========================================
# 1. 데이터 로드 및 피처 엔지니어링
# ==========================================
df = pd.read_csv('tr_data_2010_2025.csv')
df['일시'] = pd.to_datetime(df['일시'])
df = df.ffill().bfill() 

# 계절 및 시간 주기성 피처 생성
df['month'] = df['일시'].dt.month
def get_season(month):
    if month in[3, 4, 5]: return '봄'
    elif month in [6, 7, 8]: return '여름'
    elif month in[9, 10, 11]: return '가을'
    else: return '겨울'
df['계절'] = df['month'].apply(get_season)
season_dummies = pd.get_dummies(df['계절'], prefix='계절', dtype=float)
df = pd.concat([df, season_dummies], axis=1)

df['hour'] = df['일시'].dt.hour
df['dayofyear'] = df['일시'].dt.dayofyear
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
df['day_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365.25)
df['day_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365.25)
df['is_night'] = ((df['hour'] >= 18) | (df['hour'] < 9)).astype(float)
df.drop(columns=['month', '계절', 'hour', 'dayofyear'], inplace=True)

feature_cols = df.columns.drop('일시').tolist()
num_features = len(feature_cols)

# 🚨 타겟을 '기온'으로 설정
target_col =[col for col in feature_cols if '습도' in col][0] 
print(f"적용된 총 변수 개수: {num_features}개 (타겟: {target_col})")

scaler = StandardScaler()
data_scaled = scaler.fit_transform(df[feature_cols].values)

# ==========================================
# 2. 파라미터 및 모델 정의
# ==========================================
seq_len = 720  
pred_len = 744 

class iTransformer_Residual(nn.Module):
    def __init__(self, seq_len, pred_len, num_features, d_model=128, nhead=8, num_layers=2):
        super(iTransformer_Residual, self).__init__()
        self.enc_embedding = nn.Linear(seq_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4, batch_first=True, dropout=0.1
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.projector = nn.Linear(d_model, pred_len)

    def forward(self, x):
        seq_mean = torch.mean(x, dim=1, keepdim=True)
        x_res = x - seq_mean  
        x_res = x_res.transpose(1, 2)  
        x_enc = self.enc_embedding(x_res)
        enc_out = self.encoder(x_enc)
        dec_out = self.projector(enc_out)
        out_res = dec_out.transpose(1, 2)
        return out_res + seq_mean

# ==========================================
# 3. 앙상블 모델 학습 및 로드 (메모리에 캐싱)
# ==========================================
seeds =[42, 123, 7, 999, 2024]
ensemble_models =[]

class TimeSeriesDataset(Dataset):
    def __init__(self, data, seq_len, pred_len):
        self.data = data
        self.seq_len = seq_len
        self.pred_len = pred_len
    def __len__(self): return len(self.data) - self.seq_len - self.pred_len + 1
    def __getitem__(self, idx): return torch.tensor(self.data[idx:idx+self.seq_len], dtype=torch.float32), torch.tensor(self.data[idx+self.seq_len:idx+self.seq_len+self.pred_len], dtype=torch.float32)

dataset = TimeSeriesDataset(data_scaled, seq_len, pred_len)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print("\n--- 앙상블 모델(5개) 준비 중 ---")
for seed in seeds:
    model_path = f'itransformer_humidity_seed_{seed}.pth'
    model = iTransformer_Residual(seq_len=seq_len, pred_len=pred_len, num_features=num_features).to(device)
    
    # 모델 파일이 없으면 학습 진행
    if not os.path.exists(model_path):
        print(f"[{seed}] 학습된 모델이 없어 새로 학습을 시작합니다...")
        set_seed(seed)
        criterion = nn.HuberLoss(delta=0.5)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        model.train()
        for epoch in range(10): # 10 Epochs
            for batch_x, batch_y in dataloader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                optimizer.zero_grad()
                loss = criterion(model(batch_x), batch_y)
                loss.backward()
                optimizer.step()
        torch.save(model.state_dict(), model_path)
    
    # 모델 가중치 로드 후 리스트에 추가 (추론 속도 향상)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()
    ensemble_models.append(model)

print("--- 모델 준비 완료! ---")

# ==========================================
# 4. 2024년 1월 ~ 12월 월간 지표 예측 (앙상블)
# ==========================================
print("\n--- 2024년 1월 ~ 12월 월간 지표 추론 시작 ---")
temp_results =[]

for month in range(1, 13):
    target_start_date_str = f'2024-{month:02d}-01'
    target_start_date = pd.to_datetime(target_start_date_str + ' 00:00:00')

    # 데이터 존재 확인
    candidates = df[df['일시'] >= target_start_date]
    if candidates.empty:
        print(f"경고: {target_start_date_str} 이후의 데이터가 없습니다.")
        continue
        
    idx_start = candidates.index[0]
    actual_start_date = df.loc[idx_start, '일시']
    
    # 🚨[추가됨] 해당 월의 예측해야 할 총 시간 계산
    target_month_end = actual_start_date + pd.offsets.MonthEnd(0) + pd.Timedelta(hours=23)
    target_hours_count = int((target_month_end - actual_start_date).total_seconds() / 3600) + 1

    print(f"처리 중: {target_start_date_str} -> 실제 시작 데이터: {actual_start_date} ({target_hours_count}시간 예측)")

    # 입력 데이터 추출
    input_data = data_scaled[idx_start - seq_len : idx_start]
    input_tensor = torch.tensor(input_data, dtype=torch.float32).unsqueeze(0).to(device)

    # ==============================
    # 5개의 앙상블 모델 예측 수행
    # ==============================
    all_predictions =[]
    with torch.no_grad():
        for model in ensemble_models:
            pred_scaled = model(input_tensor).squeeze(0).cpu().numpy()[:target_hours_count, :]
            pred_data = scaler.inverse_transform(pred_scaled)
            all_predictions.append(pred_data)
            
    # 예측값 평균 (앙상블)
    final_pred_data = np.mean(all_predictions, axis=0)

    # DataFrame 생성 (인덱스를 datetime으로 설정해야 resample 가능)
    dates = df['일시'].iloc[idx_start : idx_start + target_hours_count].reset_index(drop=True)
    df_pred = pd.DataFrame(final_pred_data, index=dates, columns=feature_cols)

    # ==============================
    # 해당 월의 기온 지표 계산
    # ==============================
    mean_t = df_pred[target_col].mean()
    max_t = df_pred[target_col].max()
    min_t = df_pred[target_col].min()

    # 결과 리스트에 추가
    temp_results.append({
        '월': f'{month}월',
        '습도_평균': mean_t,
        '습도_최고': max_t,
        '습도_최저': min_t
    })
    
    print(f" -> [{month}월] 예측 및 지표 계산 완료")

# ==========================================
# 5. 결과를 DataFrame으로 변환 및 CSV 저장
# ==========================================
df_temp_results = pd.DataFrame(temp_results)
df_temp_results.to_csv('monthly_humidity_results_ensemble.csv', index=False, encoding='utf-8-sig')

print("\n--- 전체 처리 완료 ---")
print("결과가 'monthly_humidity_results_ensemble.csv'에 성공적으로 저장되었습니다.")
print(df_temp_results)

사용 중인 디바이스: cuda
적용된 총 변수 개수: 44개 (타겟: 습도(%))

--- 앙상블 모델(5개) 준비 중 ---
[42] 학습된 모델이 없어 새로 학습을 시작합니다...
[123] 학습된 모델이 없어 새로 학습을 시작합니다...
[7] 학습된 모델이 없어 새로 학습을 시작합니다...
[999] 학습된 모델이 없어 새로 학습을 시작합니다...
[2024] 학습된 모델이 없어 새로 학습을 시작합니다...
--- 모델 준비 완료! ---

--- 2024년 1월 ~ 12월 월간 지표 추론 시작 ---
처리 중: 2024-01-01 -> 실제 시작 데이터: 2024-01-01 01:00:00 (744시간 예측)
 -> [1월] 예측 및 지표 계산 완료
처리 중: 2024-02-01 -> 실제 시작 데이터: 2024-02-01 00:00:00 (696시간 예측)
 -> [2월] 예측 및 지표 계산 완료
처리 중: 2024-03-01 -> 실제 시작 데이터: 2024-03-01 00:00:00 (744시간 예측)
 -> [3월] 예측 및 지표 계산 완료
처리 중: 2024-04-01 -> 실제 시작 데이터: 2024-04-01 00:00:00 (720시간 예측)
 -> [4월] 예측 및 지표 계산 완료
처리 중: 2024-05-01 -> 실제 시작 데이터: 2024-05-01 00:00:00 (744시간 예측)
 -> [5월] 예측 및 지표 계산 완료
처리 중: 2024-06-01 -> 실제 시작 데이터: 2024-06-01 00:00:00 (720시간 예측)
 -> [6월] 예측 및 지표 계산 완료
처리 중: 2024-07-01 -> 실제 시작 데이터: 2024-07-01 00:00:00 (744시간 예측)
 -> [7월] 예측 및 지표 계산 완료
처리 중: 2024-08-01 -> 실제 시작 데이터: 2024-08-01 00:00:00 (744시간 예측)
 -> [8월] 예측 및 지표 계산 완료
처리 중: 2024-09-01 -> 실제 시작 데이터: 2